[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/altair-certified/notebooks/day-08-customization.ipynb#scrollTo=aa110001)

---
# Day 8 · Customization & Themes
**certified-journeys / altair-certified** · Day 8 · Themes, Axes, Legends & Export

> **Goal for today:** Control every visual aspect of an Altair chart — from built-in themes to custom reusable theme functions — and export production-ready images.

## Why Customization Matters

Altair's default theme is clean but generic. For polished reports, dashboards, or publications you need:

- **Consistent branding** across every chart in a notebook or app
- **Legible axes** — rotated labels, correct font sizes, grid lines on or off
- **Responsive widths** — charts that fill their container rather than a fixed pixel width
- **Exportable images** — PNG/SVG for slides, PDFs, and CI artifacts

Altair exposes all of this through the **`configure_*()` API** and the **`alt.theme.register()` system**.

## Customization Quick Reference

| Method | Scope | Example |
|---|---|---|
| `alt.theme.enable('fivethirtyeight')` | Global (all charts) | Built-in themes |
| `chart.configure_axis(...)` | This chart's axes | Font size, grid, label angle |
| `chart.configure_legend(...)` | This chart's legend | Position, title font |
| `chart.configure_view(...)` | The chart frame | Border, background |
| `chart.configure_mark(...)` | Mark defaults | Default color, opacity |
| `alt.theme.register(fn, name)` | Custom global theme | Reusable branded theme |

In [ ]:
%pip install -q altair vega-datasets altair-saver

## Step 1 · Built-In Themes

Altair ships with several named themes. `alt.theme.enable()` sets the active theme globally for the rest of the notebook session. Call it at the top of any notebook to establish a consistent look.

In [ ]:
import altair as alt
from vega_datasets import data

cars = data.cars()

def base_scatter(title_prefix=""):
    """Helper: returns a simple scatter with a theme-identifying title."""
    return (
        alt.Chart(cars)
        .mark_point(size=60, filled=True)
        .encode(
            x=alt.X("Horsepower:Q", scale=alt.Scale(zero=False)),
            y=alt.Y("Miles_per_Gallon:Q", scale=alt.Scale(zero=False)),
            color=alt.Color("Origin:N"),
        )
        .properties(title=title_prefix, width=340, height=220)
    )

# --- demo each built-in theme ---
themes_to_show = ["default", "fivethirtyeight", "ggplot2", "dark"]
charts = []
for t in themes_to_show:
    with alt.theme.enable(t):   # context-manager: theme applies only inside the block
        c = base_scatter(title_prefix=f"theme: {t}")
        charts.append(c)

# Display as 2x2 grid
(charts[0] | charts[1]) & (charts[2] | charts[3])

**What just happened?**

- `alt.theme.enable(name)` used as a **context manager** applies a theme only within the `with` block — very handy for side-by-side comparisons.
- Called without a context manager, it persists for all subsequent charts in the session.
- Available built-in themes: `'default'`, `'opaque'`, `'dark'`, `'fivethirtyeight'`, `'ggplot2'`, `'latimes'`, `'quartz'`, `'vox'`.
- Themes set defaults for colors, fonts, and grid lines; individual `configure_*` calls still override them.

## Step 2 · configure_axis() — Labels, Angles, Grids

`configure_axis()` targets **all axes** in the chart. You can also use `configure_axisX()` / `configure_axisY()` to target individual axes. Common properties:

| Property | Type | Effect |
|---|---|---|
| `labelFontSize` | int | Label text size |
| `labelAngle` | int (degrees) | Rotate labels (negative = counter-clockwise) |
| `grid` | bool | Show/hide grid lines |
| `titleFontSize` | int | Axis title text size |
| `tickCount` | int | Approximate tick count |
| `labelLimit` | int | Max pixels for truncating long labels |

In [ ]:
# Reset to default theme for the rest of this notebook
alt.theme.enable("default")

# Build a bar chart with long x-axis labels
movies = data.movies()

bar_raw = (
    alt.Chart(movies)
    .mark_bar()
    .encode(
        x=alt.X("Major_Genre:N", sort="-y", title="Genre"),
        y=alt.Y("mean(IMDB_Rating):Q", title="Mean IMDB Rating", scale=alt.Scale(domain=[6, 8])),
        color=alt.Color("Major_Genre:N", legend=None),
    )
    .properties(width=550, height=280)
)

bar_styled = bar_raw.configure_axis(
    labelFontSize=12,
    titleFontSize=13,
    labelAngle=-40,        # tilt x-labels for readability
    grid=False,            # remove horizontal grid lines
    tickColor="#888",
).configure_axisY(
    grid=True,             # keep y-axis grid lines
    gridDash=[4, 4],       # dashed lines look cleaner
    gridColor="#e0e0e0",
)

bar_styled

**What just happened?**

- `configure_axis()` sets defaults for **all** axes; `configure_axisX()` / `configure_axisY()` override per-axis.
- `labelAngle=-40` rotates labels 40° counter-clockwise — useful for long genre or date strings.
- `gridDash=[4, 4]` creates dashed grid lines: 4px dash, 4px gap.
- **`configure_*` methods return a new chart object** — they are immutable configuration steps, not mutations.

## Step 3 · configure_legend() — Position and Style

The legend can be repositioned, resized, or hidden. Key properties for `configure_legend()`:

| Property | Effect |
|---|---|
| `orient` | `'top'`, `'bottom'`, `'left'`, `'right'`, `'top-left'`, … |
| `titleFontSize` | Legend title text size |
| `labelFontSize` | Legend item text size |
| `columns` | Items per row (for horizontal legends) |
| `fillColor` | Legend background |
| `strokeColor` | Legend border |

In [ ]:
scatter_leg = (
    alt.Chart(cars)
    .mark_point(size=80, filled=True)
    .encode(
        x=alt.X("Horsepower:Q", scale=alt.Scale(zero=False)),
        y=alt.Y("Miles_per_Gallon:Q", scale=alt.Scale(zero=False)),
        color=alt.Color("Origin:N", title="Country of Origin"),
        shape=alt.Shape("Origin:N"),
    )
    .properties(title="Legend positioned at bottom", width=500, height=280)
    .configure_legend(
        orient="bottom",         # place legend below the chart
        columns=3,               # 3 items per row
        titleFontSize=12,
        labelFontSize=11,
        symbolSize=100,
        fillColor="#f9f9f9",
        strokeColor="#dddddd",
        padding=6,
    )
)

scatter_leg

**What just happened?**

- `orient='bottom'` moves the legend below the chart area.
- `columns=3` puts all 3 origin items side-by-side in one row.
- `symbolSize=100` makes the legend glyphs larger for better readability.
- **You can also suppress the legend inline** on a specific encoding with `legend=None`: `alt.Color('Origin:N', legend=None)`.

## Step 4 · Responsive Width with width="container"

Setting `width='container'` makes a chart fill its parent HTML element. This is essential for:

- Jupyter notebooks (chart fills the cell output area)
- HTML pages (chart fills a `<div>` whose width is set by CSS)
- Dashboards (charts resize with the browser window)

**Rule:** use `width='container'` for fluid width, and set an explicit `height` so the chart doesn't become a flat line.

In [ ]:
responsive_chart = (
    alt.Chart(cars)
    .mark_area(opacity=0.7)
    .encode(
        x=alt.X("Year:T", timeUnit="year"),
        y=alt.Y("mean(Miles_per_Gallon):Q", title="Mean MPG"),
        color=alt.Color("Origin:N"),
    )
    .properties(
        width="container",  # fills parent element width
        height=300,         # fixed height prevents zero-height rendering
        title="Average MPG over time (responsive width)"
    )
)

# Note: 'container' width renders correctly in Jupyter and HTML pages.
# In some notebook environments it shows a default pixel width — that's normal.
responsive_chart

**What just happened?**

- `width='container'` signals Vega-Lite to use 100% of the available container width.
- **Always pair it with a fixed `height`** — without it, Vega-Lite may default to a very small height.
- In a Jupyter notebook the cell output area *is* the container, so the chart fills it.
- In a `.html` file, set the parent `<div>` width with CSS and the chart will match it.

## Step 5 · Custom Reusable Theme with alt.theme.register()

A custom theme is a Python function that returns a dictionary of Vega-Lite top-level config keys. Register it with `alt.theme.register(fn, name='my-theme')` and then enable it with `alt.theme.enable('my-theme')`.

This is the right pattern for **sharing a consistent brand style** across a team or project — define it once, import it everywhere.

In [ ]:
def certified_journeys_theme():
    """Custom theme: clean, print-friendly, slightly larger text."""
    font = "DM Sans, Helvetica Neue, Arial, sans-serif"
    return {
        "config": {
            "background": "#FAFAFA",
            "view": {"stroke": "transparent"},  # remove chart border
            "title": {
                "font": font,
                "fontSize": 15,
                "fontWeight": 600,
                "color": "#18181A",
                "anchor": "start",     # left-align titles
                "offset": 6,
            },
            "axis": {
                "labelFont": font,
                "labelFontSize": 11,
                "titleFont": font,
                "titleFontSize": 12,
                "titleColor": "#52524E",
                "gridColor": "#E8E6DF",
                "gridWidth": 0.75,
                "tickColor": "#AAAAAA",
                "domainColor": "#CCCCCC",
            },
            "legend": {
                "labelFont": font,
                "labelFontSize": 11,
                "titleFont": font,
                "titleFontSize": 11,
                "titleFontWeight": 600,
            },
            "range": {
                # Custom categorical palette
                "category": ["#378ADD", "#E8890C", "#3BB36B", "#D85A30", "#7F77DD", "#0891B2"]
            },
        }
    }

# Register the theme so Altair knows about it
alt.theme.register(certified_journeys_theme, name="certified-journeys")

# Enable it globally for the rest of this notebook
alt.theme.enable("certified-journeys")

# Test: any new chart will now use this theme
theme_test = (
    alt.Chart(cars)
    .mark_bar()
    .encode(
        x=alt.X("Origin:N"),
        y=alt.Y("mean(Horsepower):Q", title="Mean Horsepower"),
        color=alt.Color("Origin:N"),
    )
    .properties(title="Custom Theme Applied", width=400, height=250)
)

theme_test

**What just happened?**

- The theme function returns a dict with a top-level `'config'` key — that's the Vega-Lite config object.
- `alt.theme.register(fn, name='...')` stores the function; `alt.theme.enable('...')` activates it.
- **`range.category`** defines the default categorical color palette — our 6-color brand palette replaces Altair's defaults.
- `view.stroke: transparent` removes the thin border Altair draws around the chart area by default.

## Step 6 · Export to PNG with altair-saver

`altair-saver` extends Altair with `.save()` support for PNG, SVG, and PDF. It uses a headless browser (via `vl-convert`) under the hood — no separate Chrome installation needed with modern versions.

**Production equivalent:** in CI pipelines, use `vl-convert` directly as a CLI tool for batch chart rendering.

> Note: PNG export in Colab may require additional runtime permissions; the code below handles this gracefully.

In [ ]:
import os

export_chart = (
    alt.Chart(cars)
    .mark_point(size=80, filled=True)
    .encode(
        x=alt.X("Horsepower:Q", scale=alt.Scale(zero=False)),
        y=alt.Y("Miles_per_Gallon:Q", scale=alt.Scale(zero=False)),
        color=alt.Color("Origin:N"),
        tooltip=["Name:N", "Horsepower:Q", "Miles_per_Gallon:Q"],
    )
    .properties(title="Export Demo — Cars Dataset", width=500, height=300)
)

# Save as JSON spec (always works — no headless browser needed)
export_chart.save("chart_spec.json")
print("Saved chart_spec.json")

# Save as SVG via vl-convert (included in altair-saver)
try:
    export_chart.save("chart_output.svg")
    size_kb = os.path.getsize("chart_output.svg") / 1024
    print(f"Saved chart_output.svg ({size_kb:.1f} KB)")
except Exception as e:
    print(f"SVG export requires vl-convert: {e}")
    print("Install with: pip install vl-convert-python")

# Save as PNG (requires vl-convert)
try:
    export_chart.save("chart_output.png", scale_factor=2.0)  # 2x for retina
    size_kb = os.path.getsize("chart_output.png") / 1024
    print(f"Saved chart_output.png ({size_kb:.1f} KB)")
except Exception as e:
    print(f"PNG export requires vl-convert: {e}")

# Display the chart inline regardless of export success
export_chart

**What just happened?**

- `chart.save('file.json')` exports the raw Vega-Lite spec — always works, no extra dependencies.
- `chart.save('file.svg')` and `chart.save('file.png')` require `vl-convert-python` (installed by `altair-saver`).
- `scale_factor=2.0` doubles the resolution for retina / print-quality output.
- **For batch export in CI:** `vl-convert vl2png --input spec.json --output chart.png` from the command line.

In [ ]:
# Challenge: Build a polished, themed bar chart
# Requirements:
#   1. Use the gapminder dataset from vega_datasets
#   2. Show mean life expectancy by continent/cluster (use 'cluster' column)
#   3. Apply the 'certified-journeys' theme registered earlier in this notebook
#      (or define and register your own custom theme)
#   4. Configure the axis: no grid lines on x, dashed grid on y, label angle 0
#   5. Move the legend to 'top'
#   6. Set width='container', height=300
#   7. Add a descriptive title
#   8. Try saving it as a JSON spec file

from vega_datasets import data as vd

gap = vd.gapminder()
print(gap.columns.tolist())

# TODO: ensure the 'certified-journeys' theme is enabled
# alt.theme.enable('certified-journeys')

# TODO: build the chart
# polished_bar = (
#     alt.Chart(gap)
#     .mark_bar()
#     .encode(...)
#     .properties(width='container', height=300, title='...')
#     .configure_axis(...)
#     .configure_legend(orient='top')
# )

# TODO: display
# polished_bar

# TODO: save spec
# polished_bar.save('polished_bar.json')

---
## Day 8 key concepts recap

| Concept | What to remember |
|---|---|
| `alt.theme.enable(name)` | Activates a named theme globally or in a `with` block |
| Built-in themes | `default`, `dark`, `fivethirtyeight`, `ggplot2`, `latimes`, `quartz`, `vox` |
| `configure_axis()` | Sets label size, angle, grid lines for all axes |
| `configure_axisX/Y()` | Overrides per-axis; takes same properties as `configure_axis` |
| `configure_legend()` | Controls position (`orient`), size, columns, background |
| `width='container'` | Responsive width; always pair with explicit `height` |
| `alt.theme.register(fn, name)` | Register custom theme; fn returns `{'config': {...}}` |
| `chart.save('file.png')` | Export PNG/SVG via `vl-convert-python` |

> **Tip:** Set `width='container'` to make charts fill their parent element in Jupyter or HTML pages. Combine with `chart.properties(height=300)` for a fixed-height responsive chart.

---
## What's next
**Day 9** → Geographic Visualization — render world maps with TopoJSON, build choropleth maps with `transform_lookup`, and layer city markers on a projected map.

Mark Day 8 complete in your [tracker](../index.html).